# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Published:", getattr(metadata, 'datePublished', 'N/A'))
print("Version:", getattr(metadata, 'version', 'N/A'))
print("License:", getattr(metadata, 'license', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their `@id`s, using metadata from the Croissant schema.

In [ ]:
# List available record sets and their fields using their @id identifiers
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    print("Available Record Sets:")
    for rs in metadata.record_sets:
        print(f"- Record Set Name: {rs.name}\n  @id: {rs.id_}")
        if hasattr(rs, 'fields') and rs.fields:
            print("  Fields:")
            for f in rs.fields:
                print(f"    - {f.name} (@id: {f.id_}) | dataType: {getattr(f, 'data_type', 'N/A')}")
        print()
else:
    print("No record sets discovered in metadata. Attempting to infer record sets from schema.")

# For compatibility, list available data distributions (files with data) and try to list columns if possible
if hasattr(metadata, 'distributions') and metadata.distributions:
    print("Available Data Distributions (files):")
    for dist in metadata.distributions:
        print(f"- Name: {getattr(dist, 'name', '')}\n  @id: {getattr(dist, 'id_', '')}\n  URL: {getattr(dist, 'content_url', '')}")
        if hasattr(dist, 'columns') and dist.columns:
            print("  Columns:")
            for col in dist.columns:
                print(f"    - {col.name} (@id: {col.id_}) | dataType: {getattr(col, 'data_type', 'N/A')}")
        print()
else:
    print("No distributions with columns discovered. See documentation for your data structure.")

## 3. Data Extraction

Load data from a specific record set or distribution into a DataFrame for analysis. All selections use the record set or field `@id`s as referenced above.

In [ ]:
#---
# Discover record set or distribution identifiers. For this dataset, we will extract the primary record set containing protein abundance and related info.
# You can update `primary_record_set_id` (below) to the specific value from the overview output.
#---
from pprint import pprint

# If no record_sets are available, we try the first distribution as a fallback
record_set_ids = []
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    record_set_ids = [rs.id_ for rs in metadata.record_sets]
elif hasattr(metadata, 'distributions') and metadata.distributions:
    # fallback: treat each distribution as a record set
    record_set_ids = [dist.id_ for dist in metadata.distributions]

# Print which record sets will be loaded
print("Record sets used for loading:")
pprint(record_set_ids)

dataframes = {}
for rset_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rset_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rset_id] = df
            print(f"Loaded DataFrame for record set '@id': {rset_id}")
            print(df.columns.tolist())
            display(df.head(3))
        else:
            print(f"No records found for record set '@id': {rset_id}")
    except Exception as e:
        print(f"Error loading record set '{rset_id}':", e)

# Select a primary record set for EDA (the first one by default)
if record_set_ids:
    primary_record_set_id = record_set_ids[0]
else:
    primary_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and grouping. All columns are referenced by their `@id`s as available in the DataFrame.

In [ ]:
df = dataframes[primary_record_set_id]

# Print all available columns with their corresponding field or column @id
print("Available DataFrame fields/columns (use these for numeric_field and group_field):")
print(df.columns.tolist())

### Choose a numeric column for filtering/normalization, and a categorical/group field
# Replace '<field_id>' below with a real column name/@id from your schema above (e.g., 'cr:abundance', 'cr:MW')
numeric_field_id = None
group_field_id = None

# Heuristically pick common column id's for demonstration
for col in df.columns:
    if 'abundance' in col.lower() or 'mw' in col.lower() or 'peptide' in col.lower():
        numeric_field_id = col
        break
# Fallback if not found
if numeric_field_id is None:
    # Just pick the first numeric-looking column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
# Look for a grouping field
for col in df.columns:
    if 'description' in col.lower() or 'accession' in col.lower() or 'sample' in col.lower():
        group_field_id = col
        break
if group_field_id is None and len(df.columns) > 1:
    group_field_id = df.columns[1]

print(f"Chosen numeric field for analysis: {numeric_field_id}")
print(f"Chosen grouping field: {group_field_id}")

# Remove outliers and NaNs before analysis
eda_df = df.copy()
if numeric_field_id is not None:
    eda_df = eda_df[pd.to_numeric(eda_df[numeric_field_id], errors='coerce').notna()]
    eda_df[numeric_field_id] = pd.to_numeric(eda_df[numeric_field_id], errors='coerce')
    # Simple outlier filter
    threshold = eda_df[numeric_field_id].mean() + 3 * eda_df[numeric_field_id].std()
    filtered_df = eda_df[eda_df[numeric_field_id] < threshold]
    print(f"Filtered records: {len(filtered_df)} remain after removing {numeric_field_id} > {threshold:.2f}")

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by field
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    # Histogram
    sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True, ax=axes[0])
    axes[0].set_title(f"Distribution of {numeric_field_id}")

    # Boxplot by group if categorical available and not too many groups
    if group_field_id:
        n_groups = filtered_df[group_field_id].nunique()
        if n_groups <= 20:
            sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df, ax=axes[1])
            axes[1].set_title(f"{numeric_field_id} by {group_field_id}")
            axes[1].tick_params(labelrotation=45)
        else:
            sns.violinplot(y=numeric_field_id, data=filtered_df, ax=axes[1])
            axes[1].set_title(f"Distribution of {numeric_field_id}")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we've used the `mlcroissant` library to load and explore the FAIR^2 protein abundance dataset. We reviewed the data structure, loaded records by `@id`, performed basic EDA with normalization and filtering, and visualized value distributions. Use the column and field `@id`s for robust automated workflows, and consult the Croissant schema for details on additional fields and record sets. Further, domain-specific analyses (e.g., protein-coding, modification annotations) can now be performed in follow-up notebooks.